# INSCO bilingual LoRA experiment (Google Drive + Colab)

This notebook reads the private JSONL files from the Drive root, fine-tunes an **adapter** (not full model weights) for `Qwen/Qwen2.5-1.5B-Instruct`, evaluates only on the held-out validation file, and saves a timestamped adapter folder back to Drive. It does not upload data elsewhere except downloading the chosen public base model to the Colab runtime.

**Important:** use only after reviewing the training examples and rights. This is a small educational experiment, not engineering authorization or code acceptance. Free Colab GPU access is limited and may be unavailable. No paid service is called by this notebook.

In [ ]:
%pip -q install transformers==4.49.0 datasets==3.3.2 peft==0.14.0 trl==0.15.2 accelerate==1.3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, math, time, torch

DRIVE_ROOT = Path('/content/drive/MyDrive')
TRAIN_FILE = DRIVE_ROOT / 'INSCO_AI_Kaggle_Train.jsonl'
VALID_FILE = DRIVE_ROOT / 'INSCO_AI_Kaggle_Validation.jsonl'
assert TRAIN_FILE.is_file(), f'Missing: {TRAIN_FILE}'
assert VALID_FILE.is_file(), f'Missing: {VALID_FILE}'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then rerun.'
print(torch.cuda.get_device_name(0), 'train bytes:', TRAIN_FILE.stat().st_size, 'validation bytes:', VALID_FILE.stat().st_size)


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
raw = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VALID_FILE)})

def check_example(example):
    messages = example.get('messages')
    assert isinstance(messages, list) and [m.get('role') for m in messages] == ['system', 'user', 'assistant']
    assert all(isinstance(m.get('content'), str) and m['content'].strip() for m in messages)
    return example
raw = raw.map(check_example)
print(raw)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_chat(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}
rendered = raw.map(render_chat, remove_columns=raw['train'].column_names)
print('Rendered example chars:', len(rendered['train'][0]['text']))


In [ ]:
use_bf16 = bool(torch.cuda.is_bf16_supported())
torch_dtype = torch.bfloat16 if use_bf16 else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch_dtype, device_map='auto')
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
run_dir = Path('/content/insco_lora_run')
args = SFTConfig(
    output_dir=str(run_dir), num_train_epochs=3,
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_ratio=0.05,
    logging_steps=5, eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    fp16=not use_bf16, bf16=use_bf16, gradient_checkpointing=True,
    max_seq_length=2048, dataset_text_field='text', packing=False, report_to='none',
)
trainer = SFTTrainer(
    model=model, args=args, train_dataset=rendered['train'], eval_dataset=rendered['validation'],
    processing_class=tokenizer, peft_config=lora,
)
print('Trainable parameters:', sum(p.numel() for p in trainer.model.parameters() if p.requires_grad))


In [ ]:
train_result = trainer.train()
metrics = trainer.evaluate()
metrics['train_runtime_seconds'] = float(train_result.metrics.get('train_runtime', 0))
metrics['validation_perplexity_including_prompt_tokens'] = math.exp(metrics['eval_loss']) if metrics.get('eval_loss', 99) < 20 else None
print(json.dumps(metrics, indent=2))


In [ ]:
# Qualitative spot-checks on held-out inputs. Review these before considering deployment.
from datetime import datetime

def generate_answer(example):
    messages = example['messages'][:-1]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    device = trainer.model.get_input_embeddings().weight.device
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = trainer.model.generate(**inputs, max_new_tokens=180, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

spot_checks = []
for index in range(min(8, len(raw['validation']))):
    example = raw['validation'][index]
    spot_checks.append({
        'index': index, 'language': example.get('language', 'see prompt'),
        'prompt': example['messages'][1]['content'][-900:],
        'expected': example['messages'][2]['content'],
        'generated': generate_answer(example),
    })
run_name = datetime.utcnow().strftime('run_%Y%m%d_%H%M%S')
out_dir = DRIVE_ROOT / 'INSCO_AI_LoRA_Runs' / run_name
out_dir.mkdir(parents=True, exist_ok=False)
trainer.model.save_pretrained(out_dir / 'adapter')
tokenizer.save_pretrained(out_dir / 'adapter')
(out_dir / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
(out_dir / 'heldout_spot_checks.json').write_text(json.dumps(spot_checks, ensure_ascii=False, indent=2), encoding='utf-8')
(out_dir / 'model_card.txt').write_text(
    f'Base model: {MODEL_ID}\nAdapter: LoRA, rank 8\nDataset: INSCO authored bilingual lessons; 160 train / 32 held-out validation examples.\n'
    'This experimental adapter is not an approved NDT/code authority. Review outputs and licenses before any deployment.', encoding='utf-8')
print('Saved adapter and evaluation files to:', out_dir)
print('Inspect heldout_spot_checks.json. Do not deploy unless technical review and the full INSCO regression suite pass.')


## After the run

The Drive folder `INSCO_AI_LoRA_Runs/run_.../adapter` contains the LoRA adapter, not a standalone merged model. Record Colab GPU/runtime, package versions, and metrics. Check held-out Arabic and English answers manually, especially whether it invents code limits. A good loss value is not proof of domain accuracy. Do not point production INSCO at this adapter without a separate serving path, security review, source review, and regression tests.